แล้วตอนนี้ Dataset ของคุณ "ขาด" อะไรบ้าง? (สิ่งที่ต้องทำก่อนรันโค้ดเทรน)
ก่อนที่คุณจะได้ไฟล์ตารางสวยๆ แบบด้านบนไปเสียบเข้าโค้ด นี่คือสิ่งที่คุณต้องลงมือทำ (Data Pipeline):

ขาดข้อมูลดิบ (Raw Data): คุณต้องไปดึงข้อความ (Scraping) จาก Twitter, YouTube หรือ Pantip มาเก็บไว้ในไฟล์ Excel ก่อน (ตั้งเป้าไว้ที่ 2,000 - 5,000 ข้อความ)

ขาดการทำความสะอาด (Preprocessing): ข้อความที่ดึงมาจะมีแท็ก HTML, ลิงก์ขยะ, หรือเว้นวรรคแปลกๆ คุณต้องเขียนสคริปต์ Python ใช้ PyThaiNLP เพื่อล้างข้อมูลขยะพวกนี้ออก

ขาดการติดป้ายอารมณ์ (Labeling): นี่คืองานใช้แรงงาน (Manual work) คุณและเพื่อนต้องมานั่งอ่านข้อความทีละบรรทัด แล้วติ๊กเลือกว่าประโยคนี้คืออารมณ์อะไร (จะใช้โปรแกรม Label Studio ช่วย หรือทำใน Google Sheets / Excel แล้วติ๊ก 0 กับ 1 ตามคอลัมน์ก็ได้)

ขาดการแบ่งชุดข้อมูล (Train/Test Split): เมื่อได้ข้อมูลครบแล้ว ต้องแบ่งข้อมูลเป็นชุดสำหรับให้โมเดลเรียนรู้ (Train 80%) และชุดสำหรับทำข้อสอบวัดความแม่นยำ (Test 20%)

สรุปคือ โค้ดโมเดลน่ะพร้อมรอรับข้อมูลแล้วครับ แต่ตัวข้อมูลเองต้องผ่านการ "ดึง -> ล้าง -> นั่งอ่านแล้วติดป้ายอารมณ์" ให้เสร็จออกมาเป็นไฟล์ CSV ก่อน ถึงจะเอาไปรันเทรนโมเดลได้จริงครับ

In [1]:
pip install google-play-scraper pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 631.5 kB/s  0:00:17m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 392.4 kB/s  0:00:40m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [4]:
from google_play_scraper import reviews, Sort
import pandas as pd

print("กำลังดึงข้อมูล... อาจใช้เวลาสักครู่")

# com.x.android
# ดึงข้อมูลรีวิวจากแอป (ระบุ App ID)
result, continuation_token = reviews(
    'com.shopee.th',          # App ID (ดูได้จาก URL ของแอปบน Play Store)
    lang='th',                # ภาษาไทย
    country='th',             # สโตร์ประเทศไทย
    sort=Sort.NEWEST,         # เรียงจากใหม่ล่าสุด
    count=1000                # จำนวนที่ต้องการดึง (ดึงได้สูงสุดหลักหมื่น)
)

# นำข้อมูลที่ได้มาแปลงเป็นตาราง (DataFrame)
df = pd.DataFrame(result)

# เลือกเฉพาะคอลัมน์ 'content' (ข้อความรีวิว) และ 'score' (คะแนนดาว)
df = df[['content', 'score']]
df.rename(columns={'content': 'text'}, inplace=True)

# แสดงผล 5 บรรทัดแรก
print(df.head())

# บันทึกเป็นไฟล์ CSV นำไปเปิดใน Excel เพื่อติดป้ายอารมณ์ต่อได้เลย
df.to_csv("thai_raw_dataset_shopee.csv", index=False, encoding='utf-8-sig')
print("\nบันทึกไฟล์ thai_raw_dataset.csv สำเร็จ!")

กำลังดึงข้อมูล... อาจใช้เวลาสักครู่
                                                text  score
0                                           ใช้ได้ดี      5
1                                             โอเคดี      5
2                     ง่ายดี รอแอพไทยทำได้แบบนี้บ้าง      5
3  แอพดีทุกอย่าง แต่อยากให้เพิ่มโหมดมืดให้หน่อยค่...      3
4  เป็นแอปที่มีคุณภาฟและเป็นโปรแกรมทีให้ความเรียบ...      5

บันทึกไฟล์ thai_raw_dataset.csv สำเร็จ!
